# EC2 Code Checks Demonstration

This notebook demonstrates the three main code checks for reinforced concrete sections:

1. **BendingCheck** (EC2 §6.1) - Flexural capacity using M-N interaction diagram
2. **ShearCheck** (EC2 §6.2) - Variable strut inclination method for shear
3. **CrackingCheck** (EC2 §7.3) - Serviceability crack width calculation

All checks use first-principles strain compatibility and force equilibrium, implemented using fibre-based integration.

In [1]:
# Imports
import numpy as np
import plotly.graph_objects as go

# Section geometry
from materials.reinforced_concrete.geometry import (
    create_rectangular_section,
    create_linear_rebar_layer,
)

# Materials
from materials.reinforced_concrete.materials import (
    ConcreteMaterial,
    Rebar,
    ShearRebar,
)

# Code checks
from materials.reinforced_concrete.code_checks.ec2_2004 import (
    BendingCheck,
    ShearCheck,
    ShearLoadCase,
    CrackingCheck,
    LoadDuration,
)

## 1. Create a Reinforced Concrete Section

We'll create a typical 300mm x 600mm rectangular beam section with:
- 3H25 tension reinforcement (bottom)
- 2H16 compression reinforcement (top)
- H10@200 shear links

In [ ]:
# Section dimensions
width = 300  # mm
height = 600  # mm
cover = 35   # mm (to links, not the longitudinal bars)

# Create section
section = create_rectangular_section(
    width=width,
    height=height,
    section_name="300x600 Beam",
)

# Define reinforcement
link_diameter = 10
tension_bar_diameter = 25
compression_bar_diameter = 16

# Calculate bar positions (y-coordinate from bottom)
y_tension = cover + link_diameter + tension_bar_diameter / 2
y_compression = height - cover - link_diameter - compression_bar_diameter / 2

# Side cover for main bars
side_cover = cover + link_diameter

# Tension reinforcement (3H25 at bottom) - using create_linear_rebar_layer
tension_rebar = Rebar(diameter=tension_bar_diameter)
tension_layer = create_linear_rebar_layer(
    rebar=tension_rebar,
    n_bars=3,
    start_point=(side_cover + tension_bar_diameter / 2, y_tension),
    end_point=(width - side_cover - tension_bar_diameter / 2, y_tension),
)
section.add_rebar_group(tension_layer)

# Compression reinforcement (2H16 at top)
compression_rebar = Rebar(diameter=compression_bar_diameter)
compression_layer = create_linear_rebar_layer(
    rebar=compression_rebar,
    n_bars=2,
    start_point=(side_cover + compression_bar_diameter / 2, y_compression),
    end_point=(width - side_cover - compression_bar_diameter / 2, y_compression),
)
section.add_rebar_group(compression_layer)

# Shear reinforcement (H10@200, 2-leg links)
shear_rebar = ShearRebar(
    diameter=link_diameter,
    spacing=200,
    n_legs=2,
)

# Define concrete material
concrete = ConcreteMaterial(grade="C30/37")

section.plot()

print(f"Section: {section.section_name}")
print(f"Dimensions: {width}mm x {height}mm")
print(f"Concrete: {concrete.grade} (f_ck = {concrete.f_ck} MPa, f_cd = {concrete.f_cd:.2f} MPa)")
print(f"Tension steel: 3H{tension_bar_diameter} (A_s = {3 * tension_rebar.area:.0f} mm²)")
print(f"Compression steel: 2H{compression_bar_diameter} (A_s' = {2 * compression_rebar.area:.0f} mm²)")
print(f"Shear links: H{link_diameter}@{shear_rebar.spacing} ({shear_rebar.n_legs}-leg)")
print(f"Reinforcement ratio: {section.reinforcement_ratio:.3%}")

Section: 300x600 Beam
Dimensions: 300mm x 600mm
Concrete: C30/37 (f_ck = 30.0 MPa, f_cd = 17.00 MPa)
Tension steel: 3H25 (A_s = 1473 mm²)
Compression steel: 2H16 (A_s' = 402 mm²)
Shear links: H10@200.0 (2-leg)
Reinforcement ratio: 1.042%


---
## 2. Bending Check (EC2 §6.1)

The `BendingCheck` uses an M-N interaction diagram to check flexural capacity.

### 2.1 Create the Check and Visualize the M-N Diagram

In [3]:
# Create bending check (diagram is built on initialization)
bending_check = BendingCheck(
    section=section,
    concrete=concrete,
)

# Plot M-N interaction diagram
bending_check.plot_mn(
    title="M-N Interaction Diagram - 300x600 Beam",
    show_metadata=True,
    show=False,
)

### 2.2 Check Multiple Load Cases

In [4]:
# Define load cases to check
load_cases = [
    {"name": "LC1 - Pure bending", "M_Ed": 250, "N_Ed": 0},
    {"name": "LC2 - Bending + compression", "M_Ed": 200, "N_Ed": 500},
    {"name": "LC3 - Bending + tension", "M_Ed": 150, "N_Ed": -200},
    {"name": "LC4 - High utilization", "M_Ed": 350, "N_Ed": 100},
]

print("Bending Check Results (EC2 §6.1)")
print("=" * 70)

results = []
for lc in load_cases:
    result = bending_check.perform_check(M_Ed=lc["M_Ed"], N_Ed=lc["N_Ed"])
    results.append({**lc, "result": result})
    
    status = result.status.value.upper()
    print(f"\n{lc['name']}:")
    print(f"  M_Ed = {lc['M_Ed']:6.1f} kN·m, N_Ed = {lc['N_Ed']:6.1f} kN")
    print(f"  M_Rd = {result.details['M_Rd']:6.1f} kN·m, N_Rd = {result.details['N_Rd']:6.1f} kN")
    print(f"  Utilization: {result.utilization:.1%} [{status}]")

Bending Check Results (EC2 §6.1)

LC1 - Pure bending:
  M_Ed =  250.0 kN·m, N_Ed =    0.0 kN
  M_Rd =  320.4 kN·m, N_Rd =    0.0 kN
  Utilization: 78.0% [PASS]

LC2 - Bending + compression:
  M_Ed =  200.0 kN·m, N_Ed =  500.0 kN
  M_Rd =  406.0 kN·m, N_Rd = 1014.9 kN
  Utilization: 49.3% [PASS]

LC3 - Bending + tension:
  M_Ed =  150.0 kN·m, N_Ed = -200.0 kN
  M_Rd =  250.9 kN·m, N_Rd = -334.5 kN
  Utilization: 59.8% [PASS]

LC4 - High utilization:
  M_Ed =  350.0 kN·m, N_Ed =  100.0 kN
  M_Rd =  338.6 kN·m, N_Rd =   96.8 kN
  Utilization: 103.4% [FAIL]


### 2.3 Visualize Load Cases on M-N Diagram

In [5]:
# Plot diagram with load points
plot_points = [
    {"N_Ed": lc["N_Ed"], "M_Ed": lc["M_Ed"], "name": lc["name"]}
    for lc in load_cases
]

bending_check.plot_mn(
    load_points=plot_points,
    show_vectors=True,
    title="M-N Diagram with Load Cases",
    show=False
)

### 2.4 Stress-Strain Visualization

Visualize the strain profile and stress distribution for a specific load case.

In [6]:
# Visualize stress-strain for LC1 (pure bending)
bending_check.plot_stress_strain(
    M_Ed=250,
    N_Ed=100,
    title="Stress-Strain Distribution (M=250 kN·m, N=0)",
    section_render="filled",
    height=800,
    show=False,
)

### 2.5 Get Moment Capacity at a Given Axial Force

In [7]:
# Get moment capacity for different axial loads
axial_loads = [0, 200, 500, 1000, -100, -300]

print("Moment Capacity at Various Axial Loads")
print("=" * 50)
print(f"{'N_Ed (kN)':>12} | {'M_Rd+ (kN·m)':>14} | {'M_Rd- (kN·m)':>14}")
print("-" * 50)

for N_Ed in axial_loads:
    M_pos, M_neg = bending_check.get_moment_capacity(N_Ed=N_Ed)
    M_pos_str = f"{M_pos:.1f}" if M_pos is not None else "N/A"
    M_neg_str = f"{M_neg:.1f}" if M_neg is not None else "N/A"
    print(f"{N_Ed:>12.0f} | {M_pos_str:>14} | {M_neg_str:>14}")

Moment Capacity at Various Axial Loads
   N_Ed (kN) |   M_Rd+ (kN·m) |   M_Rd- (kN·m)
--------------------------------------------------
           0 |          320.4 |          -98.0
         200 |          355.6 |         -146.0
         500 |          394.0 |         -217.1
        1000 |          407.8 |         -328.2
        -100 |          300.4 |          -73.9
        -300 |          258.2 |          -25.9


---
## 3. Shear Check (EC2 §6.2)

The `ShearCheck` implements the Variable Strut Inclination Method for shear design.

### 3.1 Create the Check

In [8]:
# Create shear check with rigorous mode (accurate lever arm from force centroids)
shear_check = ShearCheck(
    section=section,
    concrete=concrete,
    shear_reinforcement=shear_rebar,
    use_rigorous=True,  # Use accurate lever arm calculation
)

print(f"Shear Check Configuration")
print(f"=" * 40)
print(f"Web breadth b_w: {shear_check.breadth:.0f} mm")
print(f"Design concrete strength f_cd: {shear_check.f_cd_design:.2f} MPa")
print(f"Design yield strength f_ywd: {shear_check.f_ywd_design:.0f} MPa")
print(f"Mode: {'Rigorous' if shear_check.use_rigorous else 'Approximate'} (z = {'computed' if shear_check.use_rigorous else '0.9d'})")

Shear Check Configuration
Web breadth b_w: 300 mm
Design concrete strength f_cd: 17.00 MPa
Design yield strength f_ywd: 435 MPa
Mode: Rigorous (z = computed)


### 3.2 Check Multiple Shear Load Cases

In [9]:
# Define shear load cases
shear_load_cases = [
    {"name": "Support - high shear", "V_Ed": 250, "M_Ed": 50, "N_Ed": 0},
    {"name": "Midspan - low shear", "V_Ed": 80, "M_Ed": 300, "N_Ed": 0},
    {"name": "Column face", "V_Ed": 300, "M_Ed": 100, "N_Ed": 500},
    {"name": "Tension member", "V_Ed": 150, "M_Ed": 100, "N_Ed": -200},
]

print("Shear Check Results (EC2 §6.2)")
print("=" * 90)

for lc in shear_load_cases:
    load_case = ShearLoadCase(V_Ed=lc["V_Ed"], M_Ed=lc["M_Ed"], N_Ed=lc["N_Ed"])
    result = shear_check.perform_check(load_case=load_case)
    
    status = result.status.value.upper()
    d = result.details
    
    print(f"\n{lc['name']}:")
    print(f"  V_Ed = {lc['V_Ed']:5.0f} kN, M_Ed = {lc['M_Ed']:5.0f} kN·m, N_Ed = {lc['N_Ed']:5.0f} kN")
    print(f"  V_Rd,c = {d['V_Rd_c']:5.1f} kN (concrete)")
    print(f"  V_Rd,s = {d['V_Rd_s']:5.1f} kN (reinforcement)")
    print(f"  V_Rd,max = {d['V_Rd_max']:5.1f} kN (strut crushing)")
    print(f"  Governing: {d['governing_mode']}")
    print(f"  θ = {d['theta_deg']:.1f}° (cot θ = {d['cot_theta']:.2f})")
    print(f"  Utilization: {result.utilization:.1%} [{status}]")

Shear Check Results (EC2 §6.2)

Support - high shear:
  V_Ed =   250 kN, M_Ed =    50 kN·m, N_Ed =     0 kN
  V_Rd,c =  94.3 kN (concrete)
  V_Rd,s = 408.4 kN (reinforcement)
  V_Rd,max = 644.2 kN (strut crushing)
  Governing: shear reinforcement
  θ = 21.8° (cot θ = 2.50)
  Utilization: 61.2% [PASS]

Midspan - low shear:
  V_Ed =    80 kN, M_Ed =   300 kN·m, N_Ed =     0 kN
  V_Rd,c =  94.3 kN (concrete)
  V_Rd,s = 401.4 kN (reinforcement)
  V_Rd,max = 633.1 kN (strut crushing)
  Governing: concrete
  θ = 21.8° (cot θ = 2.50)
  Utilization: 84.8% [PASS]

Column face:
  V_Ed =   300 kN, M_Ed =   100 kN·m, N_Ed =   500 kN
  V_Rd,c = 158.7 kN (concrete)
  V_Rd,s = 345.9 kN (reinforcement)
  V_Rd,max = 630.2 kN (strut crushing)
  Governing: shear reinforcement
  θ = 21.8° (cot θ = 2.50)
  Utilization: 86.7% [PASS]

Tension member:
  V_Ed =   150 kN, M_Ed =   100 kN·m, N_Ed =  -200 kN
  V_Rd,c =  68.6 kN (concrete)
  V_Rd,s = 416.8 kN (reinforcement)
  V_Rd,max = 616.6 kN (strut crushing)


C:\Users\user\Repo\Scripts\section_design_checks\materials\reinforced_concrete\code_checks\ec2_2004\shear_check.py:419: UserWarning:

Lever arm capped: z_mech=499.0mm > 0.9d=488.2mm. Using z=0.9d for EC2 truss model.



### 3.3 Compare Rigorous vs Approximate Mode

In [10]:
# Create approximate mode check for comparison
shear_check_approx = ShearCheck(
    section=section,
    concrete=concrete,
    shear_reinforcement=shear_rebar,
    use_rigorous=False,  # Always uses z = 0.9d
)

# Compare for a single load case
test_case = ShearLoadCase(V_Ed=250, M_Ed=200, N_Ed=100)

result_rigorous = shear_check.perform_check(load_case=test_case)
result_approx = shear_check_approx.perform_check(load_case=test_case)

print("Comparison: Rigorous vs Approximate Mode")
print("=" * 50)
print(f"Load: V_Ed={test_case.V_Ed} kN, M_Ed={test_case.M_Ed} kN·m, N_Ed={test_case.N_Ed} kN")
print()
print(f"{'Parameter':<20} | {'Rigorous':>12} | {'Approximate':>12}")
print("-" * 50)
print(f"{'Lever arm z (mm)':<20} | {result_rigorous.details['z']:>12.1f} | {result_approx.details['z']:>12.1f}")
print(f"{'V_Rd,s (kN)':<20} | {result_rigorous.details['V_Rd_s']:>12.1f} | {result_approx.details['V_Rd_s']:>12.1f}")
print(f"{'V_Rd,max (kN)':<20} | {result_rigorous.details['V_Rd_max']:>12.1f} | {result_approx.details['V_Rd_max']:>12.1f}")
print(f"{'Utilization':<20} | {result_rigorous.utilization:>11.1%} | {result_approx.utilization:>11.1%}")

Comparison: Rigorous vs Approximate Mode
Load: V_Ed=250.0 kN, M_Ed=200.0 kN·m, N_Ed=100.0 kN

Parameter            |     Rigorous |  Approximate
--------------------------------------------------
Lever arm z (mm)     |        467.7 |        488.2
V_Rd,s (kN)          |        399.3 |        416.8
V_Rd,max (kN)        |        649.2 |        677.8
Utilization          |       62.6% |       60.0%


### 3.4 Design Helper: Required Shear Reinforcement

In [11]:
# Calculate required shear reinforcement for different shear forces
shear_forces = [100, 150, 200, 250, 300, 350]

print("Required Shear Reinforcement (A_sw/s)")
print("=" * 60)
print(f"{'V_Ed (kN)':>10} | {'A_sw/s (mm²/mm)':>18} | {'Equiv. spacing (mm)':>20}")
print("-" * 60)

A_sw_link = shear_rebar.n_legs * np.pi * (shear_rebar.diameter ** 2) / 4

for V_Ed in shear_forces:
    A_sw_over_s = shear_check.get_required_shear_reinforcement(
        V_Ed=V_Ed, M_Ed=100, N_Ed=0, cot_theta=2.5
    )
    equiv_spacing = A_sw_link / A_sw_over_s if A_sw_over_s > 0 else float('inf')
    
    if A_sw_over_s == 0:
        print(f"{V_Ed:>10.0f} | {'Not required':>18} | {'N/A':>20}")
    else:
        print(f"{V_Ed:>10.0f} | {A_sw_over_s:>18.4f} | {equiv_spacing:>20.0f}")

Required Shear Reinforcement (A_sw/s)
 V_Ed (kN) |    A_sw/s (mm²/mm) |  Equiv. spacing (mm)
------------------------------------------------------------
       100 |             0.2629 |                  597
       150 |             0.2892 |                  543
       200 |             0.3856 |                  407
       250 |             0.4821 |                  326
       300 |             0.5785 |                  272
       350 |             0.6749 |                  233


### 3.5 Section Without Shear Reinforcement

In [12]:
# Check capacity without shear reinforcement
shear_check_unreinforced = ShearCheck(
    section=section,
    concrete=concrete,
    shear_reinforcement=None,  # No shear links
)

test_cases = [
    ShearLoadCase(V_Ed=50, M_Ed=100, N_Ed=0),
    ShearLoadCase(V_Ed=100, M_Ed=100, N_Ed=0),
    ShearLoadCase(V_Ed=80, M_Ed=50, N_Ed=300),  # Compression helps V_Rd,c
]

print("Unreinforced Shear Capacity (V_Rd,c only)")
print("=" * 60)

for tc in test_cases:
    result = shear_check_unreinforced.perform_check(load_case=tc)
    status = result.status.value.upper()
    print(f"V_Ed={tc.V_Ed:3.0f} kN, N_Ed={tc.N_Ed:4.0f} kN: "
          f"V_Rd,c={result.details['V_Rd_c']:.1f} kN, "
          f"Util={result.utilization:.1%} [{status}]")

Unreinforced Shear Capacity (V_Rd,c only)
V_Ed= 50 kN, N_Ed=   0 kN: V_Rd,c=94.3 kN, Util=53.0% [PASS]
V_Ed=100 kN, N_Ed=   0 kN: V_Rd,c=94.3 kN, Util=106.0% [FAIL]
V_Ed= 80 kN, N_Ed= 300 kN: V_Rd,c=133.0 kN, Util=60.2% [PASS]


---
## 4. Cracking Check (EC2 §7.3)

The `CrackingCheck` calculates crack widths for serviceability limit state.

### 4.1 Create the Check

In [13]:
# Create cracking check
cracking_check = CrackingCheck(
    section=section,
    concrete=concrete,
    w_k_limit=0.3,  # mm (typical for XC2/XC3 exposure)
    load_duration=LoadDuration.LONG_TERM,  # k_t = 0.4
)

print("Cracking Check Configuration")
print("=" * 40)
print(f"Crack width limit w_k,lim: {cracking_check.w_k_limit} mm")
print(f"Load duration: {cracking_check.load_duration.value} (k_t = {cracking_check.k_t})")
print(f"Bond coefficient k_1: {cracking_check.k_1}")
print(f"Modular ratio α_e: {cracking_check.alpha_e:.1f}")
print(f"Cracking moment M_cr: {cracking_check.find_cracking_moment():.1f} kN·m")

Cracking Check Configuration
Crack width limit w_k,lim: 0.3 mm
Load duration: long_term (k_t = 0.4)
Bond coefficient k_1: 0.8
Modular ratio α_e: 6.1
Cracking moment M_cr: 115.0 kN·m


### 4.2 Check Crack Widths for Multiple Load Cases

In [14]:
# Define SLS load cases (characteristic or quasi-permanent)
sls_load_cases = [
    {"name": "Low moment", "M_Ed": 80, "N_Ed": 0},
    {"name": "Service moment", "M_Ed": 150, "N_Ed": 0},
    {"name": "High moment", "M_Ed": 200, "N_Ed": 0},
    {"name": "With compression", "M_Ed": 150, "N_Ed": 200},
    {"name": "With tension", "M_Ed": 100, "N_Ed": -100},
]

M_cr = cracking_check.find_cracking_moment()

print("Cracking Check Results (EC2 §7.3)")
print(f"Cracking moment M_cr = {M_cr:.1f} kN·m")
print("=" * 80)

for lc in sls_load_cases:
    result = cracking_check.perform_check(M_Ed=lc["M_Ed"], N_Ed=lc["N_Ed"])
    d = result.details
    
    status = result.status.value.upper()
    
    print(f"\n{lc['name']}:")
    print(f"  M_Ed = {lc['M_Ed']:5.0f} kN·m, N_Ed = {lc['N_Ed']:5.0f} kN")
    
    if d.get('is_cracked', True):
        print(f"  Section is CRACKED")
        print(f"  σ_s = {d.get('sigma_s', 0):.1f} MPa (steel stress)")
        print(f"  s_r,max = {d.get('s_r_max', 0):.1f} mm (crack spacing)")
        print(f"  w_k = {d.get('w_k', 0):.3f} mm ≤ {cracking_check.w_k_limit} mm")
        print(f"  Utilization: {result.utilization:.1%} [{status}]")
    else:
        print(f"  Section is UNCRACKED (M_Ed < M_cr)")
        print(f"  w_k = 0 mm [{status}]")

Cracking Check Results (EC2 §7.3)
Cracking moment M_cr = 115.0 kN·m

Low moment:
  M_Ed =    80 kN·m, N_Ed =     0 kN
  Section is UNCRACKED (M_Ed < M_cr)
  w_k = 0 mm [PASS]

Service moment:
  M_Ed =   150 kN·m, N_Ed =     0 kN
  Section is CRACKED
  σ_s = 208.7 MPa (steel stress)
  s_r,max = 277.5 mm (crack spacing)
  w_k = 0.233 mm ≤ 0.3 mm
  Utilization: 77.5% [PASS]

High moment:
  M_Ed =   200 kN·m, N_Ed =     0 kN
  Section is CRACKED
  σ_s = 278.7 MPa (steel stress)
  s_r,max = 277.5 mm (crack spacing)
  w_k = 0.330 mm ≤ 0.3 mm
  Utilization: 109.9% [FAIL]

With compression:
  M_Ed =   150 kN·m, N_Ed =   200 kN
  Section is CRACKED
  σ_s = 149.2 MPa (steel stress)
  s_r,max = 266.4 mm (crack spacing)
  w_k = 0.148 mm ≤ 0.3 mm
  Utilization: 49.4% [PASS]

With tension:
  M_Ed =   100 kN·m, N_Ed =  -100 kN
  Section is UNCRACKED (M_Ed < M_cr)
  w_k = 0 mm [PASS]


### 4.3 Detailed Cracking Results

In [15]:
# Get detailed cracking results for a specific case
detailed = cracking_check.calculate_detailed(M_Ed=150, N_Ed=0)

print("Detailed Cracking Calculation (M_Ed = 150 kN·m)")
print("=" * 50)
print(f"Is cracked: {detailed.is_cracked}")
print(f"Neutral axis depth x: {detailed.x:.1f} mm" if detailed.x else "N/A")
print(f"Effective height h_c,ef: {detailed.h_c_ef:.1f} mm")
print(f"Tension steel area A_s: {detailed.rho_p_eff * detailed.h_c_ef * cracking_check.breadth:.0f} mm²")
print(f"Effective reinf. ratio ρ_p,eff: {detailed.rho_p_eff:.4f}")
print(f"Equivalent bar diameter φ_eq: {detailed.phi_eq:.1f} mm")
print(f"Cover to tension steel: {detailed.cover:.1f} mm")
print(f"Steel stress σ_s: {detailed.sigma_s:.1f} MPa")
print(f"Maximum crack spacing s_r,max: {detailed.s_r_max:.1f} mm")
print(f"Mean strain difference (ε_sm - ε_cm): {detailed.eps_sm_minus_eps_cm:.6f}")
print(f"Crack width w_k: {detailed.w_k:.3f} mm")
print(f"Limit w_k,lim: {detailed.w_k_limit} mm")

Detailed Cracking Calculation (M_Ed = 150 kN·m)
Is cracked: True
Neutral axis depth x: 159.0 mm
Effective height h_c,ef: 143.8 mm
Tension steel area A_s: 1473 mm²
Effective reinf. ratio ρ_p,eff: 0.0341
Equivalent bar diameter φ_eq: 25.0 mm
Cover to tension steel: 45.0 mm
Steel stress σ_s: 208.7 MPa
Maximum crack spacing s_r,max: 277.5 mm
Mean strain difference (ε_sm - ε_cm): 0.000838
Crack width w_k: 0.233 mm
Limit w_k,lim: 0.3 mm


### 4.4 Compare Short-Term vs Long-Term Loading

In [16]:
# Short-term loading (k_t = 0.6)
cracking_check_short = CrackingCheck(
    section=section,
    concrete=concrete,
    w_k_limit=0.3,
    load_duration=LoadDuration.SHORT_TERM,
)

M_test = 150  # kN·m

result_long = cracking_check.perform_check(M_Ed=M_test, N_Ed=0)
result_short = cracking_check_short.perform_check(M_Ed=M_test, N_Ed=0)

print("Effect of Load Duration on Crack Width")
print(f"M_Ed = {M_test} kN·m, N_Ed = 0")
print("=" * 50)
print(f"{'Parameter':<25} | {'Long-term':>10} | {'Short-term':>10}")
print("-" * 50)
print(f"{'k_t factor':<25} | {cracking_check.k_t:>10.1f} | {cracking_check_short.k_t:>10.1f}")
print(f"{'w_k (mm)':<25} | {result_long.details['w_k']:>10.3f} | {result_short.details['w_k']:>10.3f}")
print(f"{'Utilization':<25} | {result_long.utilization:>9.1%}  | {result_short.utilization:>9.1%}")

Effect of Load Duration on Crack Width
M_Ed = 150 kN·m, N_Ed = 0
Parameter                 |  Long-term | Short-term
--------------------------------------------------
k_t factor                |        0.4 |        0.6
w_k (mm)                  |      0.233 |      0.204
Utilization               |     77.5%  |     68.1%


### 4.5 Minimum Crack Control Reinforcement

In [17]:
# Calculate minimum reinforcement for crack control
A_s_min = cracking_check.find_minimum_crack_reinforcement(
    steel_stress=300,  # Typical SLS stress limit
    N_Ed=0,
    is_in_bending=True,
)

# Compare with provided reinforcement
A_s_provided = 3 * tension_rebar.area

print("Minimum Reinforcement for Crack Control (EC2 §7.3.2)")
print("=" * 50)
print(f"A_s,min required: {A_s_min:.0f} mm²")
print(f"A_s provided: {A_s_provided:.0f} mm²")
print(f"Ratio: {A_s_provided / A_s_min:.2f}x minimum")

Minimum Reinforcement for Crack Control (EC2 §7.3.2)
A_s,min required: 348 mm²
A_s provided: 1473 mm²
Ratio: 4.24x minimum


---
## 5. Combined Analysis: Full Design Check

Check all three criteria for a typical beam load case.

In [18]:
# Define ULS and SLS loads for a typical beam
# ULS (factored)
M_Ed_uls = 280  # kN·m
V_Ed_uls = 220  # kN
N_Ed_uls = 50   # kN (small compression)

# SLS (characteristic/quasi-permanent)
M_Ed_sls = 180  # kN·m
N_Ed_sls = 35   # kN

print("FULL DESIGN CHECK - 300x600 Beam")
print("=" * 60)
print(f"\nULS Loads: M_Ed = {M_Ed_uls} kN·m, V_Ed = {V_Ed_uls} kN, N_Ed = {N_Ed_uls} kN")
print(f"SLS Loads: M_Ed = {M_Ed_sls} kN·m, N_Ed = {N_Ed_sls} kN")

# Bending check
bending_result = bending_check.perform_check(M_Ed=M_Ed_uls, N_Ed=N_Ed_uls)
print(f"\n1. BENDING (EC2 §6.1)")
print(f"   M_Rd = {bending_result.details['M_Rd']:.1f} kN·m")
print(f"   Utilization: {bending_result.utilization:.1%} [{bending_result.status.value.upper()}]")

# Shear check
shear_result = shear_check.perform_check(
    load_case=ShearLoadCase(V_Ed=V_Ed_uls, M_Ed=M_Ed_uls, N_Ed=N_Ed_uls)
)
print(f"\n2. SHEAR (EC2 §6.2)")
print(f"   V_Rd = {shear_result.details['V_Rd']:.1f} kN ({shear_result.details['governing_mode']})")
print(f"   Utilization: {shear_result.utilization:.1%} [{shear_result.status.value.upper()}]")

# Cracking check
cracking_result = cracking_check.perform_check(M_Ed=M_Ed_sls, N_Ed=N_Ed_sls)
print(f"\n3. CRACKING (EC2 §7.3)")
print(f"   w_k = {cracking_result.details.get('w_k', 0):.3f} mm ≤ {cracking_check.w_k_limit} mm")
print(f"   Utilization: {cracking_result.utilization:.1%} [{cracking_result.status.value.upper()}]")

# Summary
all_pass = all(r.status == "pass" for r in [bending_result, shear_result, cracking_result])
print(f"\n{'=' * 60}")
print(f"OVERALL: {'ALL CHECKS PASS' if all_pass else 'SOME CHECKS FAIL'}")

FULL DESIGN CHECK - 300x600 Beam

ULS Loads: M_Ed = 280 kN·m, V_Ed = 220 kN, N_Ed = 50 kN
SLS Loads: M_Ed = 180 kN·m, N_Ed = 35 kN

1. BENDING (EC2 §6.1)
   M_Rd = 331.7 kN·m
   Utilization: 84.4% [PASS]

2. SHEAR (EC2 §6.2)
   V_Rd = 400.0 kN (shear reinforcement)
   Utilization: 55.0% [PASS]

3. CRACKING (EC2 §7.3)
   w_k = 0.276 mm ≤ 0.3 mm
   Utilization: 91.9% [PASS]

OVERALL: ALL CHECKS PASS


---
## 6. Parametric Study: Crack Width vs Moment

In [19]:
# Generate crack width curve
M_cr = cracking_check.find_cracking_moment()
moments = np.linspace(0, 250, 50)
crack_widths = []

for M in moments:
    if M <= M_cr:
        crack_widths.append(0)
    else:
        try:
            result = cracking_check.calculate_detailed(M_Ed=float(M), N_Ed=0)
            crack_widths.append(result.w_k)
        except:
            crack_widths.append(np.nan)

# Plot
fig = go.Figure()

fig.add_trace(go.Scatter(
    x=moments,
    y=crack_widths,
    mode='lines',
    name='Crack width w_k',
    line=dict(color='blue', width=2),
))

# Add limit line
fig.add_hline(
    y=cracking_check.w_k_limit,
    line_dash="dash",
    line_color="red",
    annotation_text=f"w_k,lim = {cracking_check.w_k_limit} mm",
)

# Add cracking moment line
fig.add_vline(
    x=M_cr,
    line_dash="dot",
    line_color="gray",
    annotation_text=f"M_cr = {M_cr:.0f} kN·m",
)

fig.update_layout(
    title="Crack Width vs Applied Moment",
    xaxis_title="Moment M_Ed (kN·m)",
    yaxis_title="Crack Width w_k (mm)",
    height=500,
)

fig.show()

---
## Summary

This notebook demonstrated:

1. **BendingCheck** - First-principles M-N interaction diagram for flexural design
   - `perform_check(M_Ed, N_Ed)` for utilization
   - `plot_mn()` for interactive diagrams
   - `plot_stress_strain()` for strain/stress visualization
   - `get_moment_capacity()` for design

2. **ShearCheck** - Variable strut inclination method (EC2 §6.2)
   - V_Rd,c (concrete), V_Rd,s (reinforcement), V_Rd,max (strut)
   - Rigorous vs approximate lever arm modes
   - `get_required_shear_reinforcement()` for design

3. **CrackingCheck** - Crack width calculation (EC2 §7.3)
   - Uses characteristic (SLS) material properties
   - Accounts for tension stiffening
   - Short-term vs long-term loading effects
   - Minimum reinforcement for crack control